In [1]:
! pip install ndlinear -q

In [2]:
from torchvision.datasets import CIFAR10
from torchvision import transforms
from ndlinear import NdLinear
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np
import time

In [14]:
class Model(nn.Module):
    """
    Classificador CIFAR-10 usando NdLinear para máxima eficiência de parâmetros.

    Este modelo utiliza NdLinear para preservar a estrutura multivariada das imagens
    e reduzir significativamente o número de parâmetros em comparação com nn.Linear.
    """

    def __init__(self, num_classes=10):
        super(Model, self).__init__()

        # Camada de convolução inicial para extração de features
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Batch normalization para estabilizar o treinamento
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        # Max pooling para redução de dimensionalidade
        self.pool = nn.MaxPool2d(2, 2)

        # Após 3 camadas de pooling (32->16->8->4), temos 128 canais de 4x4
        # Usando NdLinear para preservar a estrutura espacial
        self.ndlinear1 = NdLinear((128, 4, 4), (128, 2, 2))
        self.ndlinear2 = NdLinear((128, 2, 2), (64, 1, 1))

        # Camada final para classificação
        self.classifier = nn.Linear(64, num_classes)

        # Dropout para regularização
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Camadas convolucionais com ReLU e batch norm
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # NdLinear layers preservando estrutura espacial
        x = F.relu(self.ndlinear1(x))
        x = self.dropout(x)
        x = F.relu(self.ndlinear2(x))
        x = self.dropout(x)

        # Global average pooling para reduzir para 1x1
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = x.view(x.size(0), -1)

        # Classificação final
        x = self.classifier(x)
        return x

    def count_parameters(self):
        """Conta o número total de parâmetros do modelo."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_ndlinear_parameters(self):
        """Conta especificamente os parâmetros das camadas NdLinear."""
        ndlinear_params = 0
        for name, module in self.named_modules():
            if isinstance(module, NdLinear):
                ndlinear_params += sum(p.numel() for p in module.parameters() if p.requires_grad)
        return ndlinear_params

In [15]:
train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.2023, 0.1994, 0.2010))])

test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

train_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=train_transform
        )

test_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=test_transform
        )

In [16]:
batch_size = 128
train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
        )
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

In [17]:
learning_rate = 0.001
num_epochs = 150
weight_decay = 1e-4
model = Model()

In [18]:
# Otimizador e loss function
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10, verbose=True
)
criterion = nn.CrossEntropyLoss()

In [19]:
# Métricas
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_type='ndlinear'

In [20]:
def train_epoch(epoch):
        """Treina o modelo por uma época."""
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

        for _, (data, target) in enumerate(pbar):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Atualizar progress bar
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })

        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = 100. * correct / total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy)

        return epoch_loss, epoch_accuracy

In [21]:
def test_epoch():
        """Testa o modelo."""
        model.eval()
        test_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                test_loss += criterion(output, target).item()
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()

        test_loss /= len(test_loader)
        accuracy = 100. * correct / total

        test_losses.append(test_loss)
        test_accuracies.append(accuracy)

        return test_loss, accuracy

In [22]:
def train():
        """Treina o modelo completo."""
        print(f"Treinando modelo {model_type} em {device}")
        best_accuracy = 0.0

        # Move o modelo para o dispositivo (GPU ou CPU)
        model.to(device)

        print(f"Parâmetros do modelo: {model.count_parameters():,}")

        if model_type == 'ndlinear':
            ndlinear_params = model.count_ndlinear_parameters()
            print(f"Parâmetros NdLinear: {ndlinear_params:,}")

        start_time = time.time()

        for epoch in range(num_epochs):
            # Treinar
            train_loss, train_acc = train_epoch(epoch)

            # Testar
            test_loss, test_acc = test_epoch()

            # Learning rate scheduling
            scheduler.step(test_acc)

            # Salvar melhor modelo
            if test_acc > best_accuracy:
                best_accuracy = test_acc
                torch.save(model.state_dict(), f'best_model_{model_type}.pth')

            # Log para TensorBoard
            # Note: SummaryWriter should ideally be created once outside the loop
            # However, keeping it here as in the original code structure
            writer = SummaryWriter(f'runs/cifar10_{model_type}')
            writer.add_scalar('Loss/Train', train_loss, epoch)
            writer.add_scalar('Loss/Test', test_loss, epoch)
            writer.add_scalar('Accuracy/Train', train_acc, epoch)
            writer.add_scalar('Accuracy/Test', test_acc, epoch)
            writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)
            writer.close() # Close the writer after each epoch to ensure data is flushed

            print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
                  f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')

        training_time = time.time() - start_time
        print(f"\nTreinamento concluído em {training_time:.2f} segundos")
        print(f"Melhor acurácia: {best_accuracy:.2f}%")

        return {
            'best_accuracy': best_accuracy,
            'training_time': training_time,
            'final_train_loss': train_losses[-1],
            'final_test_accuracy': test_accuracies[-1]
        }

In [23]:
def plot_results():
        """Plota os resultados do treinamento."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot loss
        ax1.plot(train_losses, label='Train Loss')
        ax1.plot(test_losses, label='Test Loss')
        ax1.set_title('Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)

        # Plot accuracy
        ax2.plot(train_accuracies, label='Train Accuracy')
        ax2.plot(test_accuracies, label='Test Accuracy')
        ax2.set_title('Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig(f'training_results_{model_type}.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
metrics = train()

Treinando modelo ndlinear em cuda
Parâmetros do modelo: 119,140
Parâmetros NdLinear: 24,794


Epoch 1/150:  55%|█████▌    | 217/391 [00:13<00:10, 16.51it/s, Loss=1.6022, Acc=29.85%]

In [ ]:
plot_results()

In [ ]:
import pandas as pd

In [ ]:
# Carregar modelo treinado
model = Model()
model.load_state_dict(torch.load('best_model_ndlinear.pth', map_location='cpu'))
model.eval()

# Dataset de teste
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# Gerar predições
all_preds = []
with torch.no_grad():
    for images, _ in testloader:
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

# Criar DataFrame de submissão
df = pd.DataFrame({
    'Id': list(range(len(all_preds))),
    'Label': all_preds
})

df.to_csv('submission.csv', index=False)
print('Arquivo submission.csv gerado!')

In [ ]:
df.to_csv('submissiodont_flatten.csv', index=False)